# 00 — Data smoke-test (Phase 0)

**Purpose: prove the whole pipeline runs end-to-end on one session before we touch the science.**

The HMM at the end is *not* a result. It is a canary. If it fits, then data access,
trial alignment, binning, and model fitting all work, and Phase 1 can start.

Steps:
1. Connect to public **openalyx** (no account needed).
2. Pull the BWM metadata tables; count good units per region.
3. Find sessions where our candidate-axis regions were recorded **simultaneously**.
4. *(gated)* Download one insertion's spikes + the session trials table.
5. Build a `(trials x time x neurons)` tensor aligned to stimulus onset.
6. Fit a Poisson HMM and relate the state sequence to reaction time.

Steps 1-3 are **metadata only** and cost nothing. Step 4 downloads ~440 MB.

Run with the `neurodatarehack` conda env.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# make `src` importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.access import get_one, sessions_table, units_table, region_cooccurrence, load_units, load_trials
from src.data.build_matrices import bin_spikes_by_event, stack_for_hmmlearn, drop_silent_units

PARAMS = yaml.safe_load((ROOT / "config" / "params.yaml").read_text())
AREAS = yaml.safe_load((ROOT / "config" / "areas.yaml").read_text())
AXIS = AREAS["candidate_axis"]

rng = np.random.default_rng(PARAMS["random_seed"])
print("candidate axis:", AXIS)

## 1. Connect to openalyx

`get_one()` uses ONE-api's built-in public credentials and pins the aggregate-table
tag so `bwm_units` reproduces the paper's frozen unit set. No registration needed.

In [ ]:
one = get_one(cache_dir=PARAMS["data"]["one_cache_dir"])
print("mode:      ", one.mode)
print("cache_dir: ", one.cache_dir)

## 2. Metadata tables and good-unit yields

`units_table` asserts a hash of the unit UUIDs against the `2023_12_bwm_release`
freeze, so these are exactly the units the paper analysed.

Expect **699 insertions / 459 sessions**, and **62,990 units over 210 Beryl regions**.

In [ ]:
bwm_df = sessions_table(one)
unit_df = units_table(one)

print(f"insertions: {bwm_df.pid.nunique()}   sessions: {bwm_df.eid.nunique()}")
print(f"good units: {len(unit_df)}   Beryl regions: {unit_df.Beryl.nunique()}")

yields = pd.DataFrame({
    "good_units": unit_df[unit_df.Beryl.isin(AXIS)].Beryl.value_counts().reindex(AXIS),
    "n_sessions": unit_df[unit_df.Beryl.isin(AXIS)].groupby("Beryl", observed=True).eid.nunique().reindex(AXIS),
})
yields

## 3. Which axis regions co-occur in one session?

Single-trial methods (HMM, rSLDS, geometry, flow) need **simultaneously recorded**
populations. Pooling across sessions into a pseudopopulation destroys the trial-by-trial
correlations these models exist to capture. So the unit of selection is the *session*.

In [ ]:
sel = AREAS["selection"]
co = region_cooccurrence(
    unit_df, AXIS,
    min_units_per_region=sel["min_units_per_region"],
    min_regions_per_session=sel["min_regions_per_session"],
)
print(f"{len(co)} sessions have >= {sel['min_regions_per_session']} axis regions "
      f"with >= {sel['min_units_per_region']} good units each\n")
for _, r in co.head(6).iterrows():
    print(f"  {r.eid}  n_reg={r.n_regions}  " + ", ".join(f"{a}:{b}" for a, b in zip(r.regions, r.units)))

In [ ]:
# Pick the richest session, and within it the insertion carrying the most units
# of a single target region. That region is our smoke-test population.
best = co.iloc[0]
EID = best.eid

cand = (unit_df[(unit_df.eid == EID) & (unit_df.Beryl.isin(AXIS))]
        .groupby(["pid", "Beryl"], observed=True).size()
        .reset_index(name="n").sort_values("n", ascending=False))
display(cand)

PID = cand.iloc[0]["pid"]
REGION = cand.iloc[0]["Beryl"]
N_EXPECTED = int(cand.iloc[0]["n"])
print(f"\nsmoke-test target -> eid={EID}\n                    pid={PID}\n"
      f"                    region={REGION} ({N_EXPECTED} good units)")

## 4. Download (gated)

> **This is the only cell that downloads neural data.**
> With `qc=1.0`, `load_good_units` fetches `passingSpikes.table.pqt` (a good-units-only
> spike table) plus small `clusters.*`/`channels.*` files — **not** the raw ephys.
> Roughly **150–280 MB per insertion**. Cached afterwards, so re-running is free.

Set `DOWNLOAD = True` to proceed.

In [ ]:
DOWNLOAD = True  # <-- flip to True to fetch data

if not DOWNLOAD:
    raise RuntimeError("Set DOWNLOAD = True to download ~150-280 MB for this insertion.")

spikes, clusters = load_units(one, PID, qc=PARAMS["data"]["min_qc"])
trials, mask = load_trials(
    one, EID,
    min_rt=PARAMS["trials"]["min_rt_s"],
    max_rt=PARAMS["trials"]["max_rt_s"],
)
print(f"spikes: {spikes['times'].size:,} from {clusters.shape[0]} good units")
print(f"trials: {mask.sum()} usable of {len(trials)} total")

### Select the units belonging to our Beryl region

`clusters['acronym']` holds the *fine-grained* Allen acronym (e.g. `SCig`, `SCiw`,
`VISp2/3`). Beryl is a coarser rollup (`SCm`, `VISp`). We must map before filtering,
otherwise `acronym == 'SCm'` silently matches nothing.

In [ ]:
from iblatlas.regions import BrainRegions

br = BrainRegions()
clusters = clusters.copy()
clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")

unit_idx = np.flatnonzero((clusters["Beryl"] == REGION).values)
print(f"{len(unit_idx)} units in {REGION} (metadata said {N_EXPECTED})")
print(clusters["Beryl"].value_counts().head(8).to_string())

## 5. Build the (trials x time x neurons) tensor

Aligned to `stimOn_times`. The trial mask is applied first — it enforces the paper's
own criteria (reaction time in range, no NaN in key events).

In [ ]:
B = PARAMS["binning"]
good_trials = trials[mask]
events = good_trials[B["align_event"]].values

# keep only spikes from our region's units, and re-index those units to 0..n-1
keep_spike = np.isin(spikes["clusters"], unit_idx)
remap = -np.ones(clusters.shape[0], dtype=np.int64)
remap[unit_idx] = np.arange(unit_idx.size)

counts, bin_centers = bin_spikes_by_event(
    spike_times=spikes["times"][keep_spike],
    spike_clusters=remap[spikes["clusters"][keep_spike]],
    n_clusters=unit_idx.size,
    event_times=events,
    pre_s=B["pre_s"], post_s=B["post_s"], bin_size_s=B["bin_size_s"],
)
counts, kept = drop_silent_units(counts, min_total_spikes=1)
print(f"tensor: {counts.shape}  (trials x bins x neurons), dtype={counts.dtype}")
print(f"window: {bin_centers[0]:+.3f} .. {bin_centers[-1]:+.3f} s around {B['align_event']}")
print(f"mean rate: {counts.sum() / (counts.shape[0]*counts.shape[1]*B['bin_size_s']*counts.shape[2]):.2f} sp/s")

In [ ]:
# Sanity: population PSTH should show a visual response after stimulus onset.
psth = counts.mean(axis=(0, 2)) / B["bin_size_s"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(bin_centers, psth, lw=1.5)
ax.axvline(0, color="k", ls="--", lw=1, label="stimulus onset")
ax.set(xlabel="time from stimOn (s)", ylabel="rate (sp/s)", title=f"{REGION} population PSTH")
ax.legend(frameon=False); fig.tight_layout()

## 6. Fit a Poisson HMM (the canary)

Poisson emissions are the right likelihood for **raw spike counts** — so we deliberately
feed unsmoothed integers. `lengths` tells `hmmlearn` that trials are independent
sequences, so it never learns a transition across a trial boundary.

`n_states` here is an arbitrary Phase-0 value. Choosing K by cross-validated
log-likelihood is a **Phase 2** job — do not read anything into this fit.

In [ ]:
from hmmlearn.hmm import PoissonHMM

K = PARAMS["hmm"]["n_states"]
X, lengths = stack_for_hmmlearn(counts)

model = PoissonHMM(n_components=K, n_iter=PARAMS["hmm"]["n_iter"], random_state=PARAMS["random_seed"])
model.fit(X, lengths)

print(f"converged: {model.monitor_.converged}   iters: {model.monitor_.iter}")
print(f"log-likelihood: {model.score(X, lengths):,.1f}")

states = model.predict(X, lengths).reshape(counts.shape[0], counts.shape[1])
print(f"state sequence: {states.shape} (trials x bins)")

In [ ]:
# Do the inferred states carry anything behavioural?
# Sort trials by reaction time; if states track the decision, structure should tilt.
rt = (good_trials["firstMovement_times"] - good_trials["stimOn_times"]).values
order = np.argsort(rt)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})

im = axes[0].imshow(states[order], aspect="auto", interpolation="nearest",
                    extent=[bin_centers[0], bin_centers[-1], len(order), 0], cmap="viridis")
axes[0].plot(rt[order], np.arange(len(order)) + 0.5, color="r", lw=1.2, label="first movement")
axes[0].axvline(0, color="w", ls="--", lw=1)
axes[0].set(xlabel="time from stimOn (s)", ylabel="trial (sorted by RT)",
            title=f"{REGION}: inferred HMM state, K={K}")
axes[0].legend(frameon=False, loc="lower right")
fig.colorbar(im, ax=axes[0], label="state")

occupancy = np.stack([(states == k).mean(axis=0) for k in range(K)])
for k in range(K):
    axes[1].plot(bin_centers, occupancy[k], label=f"state {k}")
axes[1].axvline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="time from stimOn (s)", ylabel="P(state)", title="state occupancy")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()

## What passing means

If every cell above ran, then the following are all verified on real data:
openalyx access → frozen unit set → simultaneous-region selection → spike download →
trial masking → stimulus-aligned binning → Poisson HMM fit → behavioural readout.

That is Phase 0 complete. **Nothing here is a scientific result.**

Next — **Phase 1 (area selection):** widen step 3 across all 43 qualifying sessions,
decide the final region set, and freeze it into `config/areas.yaml` under `selected`.

## 7. Visual inspection of the data

The rest of the notebook is the **first look at the data** — raw rasters, population PSTHs, a couple
of `pynapple` plots, and a simple decoder — to see what the three areas actually do before any
modelling. It pools **both probes** of this session (recorded simultaneously, sharing the trial
clock) through the project's **unified preprocessing** (`src/data/preprocess.py`): `stimOn`-aligned,
`pre 0.4 / post 0.8 s`, **40 ms** bins, units ≥ 1 Hz — the exact tensor every downstream model eats.

### 7 · Population PSTH per area (juxtaposed)

Overall firing rate **averaged across trials and neurons**, one panel per area of the frozen axis
(**VISp, SCm, MRN**), juxtaposed for comparison. The **shaded band is ±1 SEM across trials** —
(std over trials)/√(n trials) of the per-trial population rate — written on each panel with that
area's own trial and unit counts, because an error band is meaningless unless you say *what it is
over*. (SEM across **trials** = how reproducible the mean response is trial to trial; a band across
**units** would answer a different question — how much units differ from each other.)


In [ ]:
# Sections 7-10 use BOTH probes (so all three regions appear) and the UNIFIED pipeline.
import pynapple as nap
from src.data.preprocess import (
    PreprocessConfig, load_session, align_times, usable_mask, preprocess_session,
)
from src.data.build_matrices import region_spike_trains, region_event_spikes
from src.viz.style import set_style, region_color, stroke, savefig, OKABE_ITO
from src.viz.nap_plots import region_crosscorrelogram, contrast_tuning

set_style()
REGIONS_3 = ("VISp", "SCm", "MRN")                 # the frozen visual->motor axis
EID = "8a3a0197-b40a-449f-be55-c00b23253bbf"       # smoke-test session (UCLA034), pinned

# frozen preprocessing: stimOn-aligned, pre 0.4 / post 0.8 s, 40 ms bins, units >= 1 Hz
cfg = PreprocessConfig(align_event="stimOn_times", pre_s=0.4, post_s=0.8,
                       bin_size_s=0.04, min_rate_hz=1.0)

trials3, mask3, probe_loads = load_session(one, EID, cfg=cfg)     # loads BOTH probes
event_times = align_times(trials3, mask3, cfg.align_event)        # stimOn of the usable trials

# trial-level covariates, kept in register with axis 0 of the tensors (same order as event_times)
um = usable_mask(trials3, mask3, cfg.align_event)  # finite stimOn -> all True for this alignment
tt = trials3[mask3][um]
choice = tt["choice"].to_numpy(float)                        # IBL: -1 / +1 / 0 (no-go)
cL = tt["contrastLeft"].to_numpy(float); cR = tt["contrastRight"].to_numpy(float)
signed_contrast = np.nan_to_num(cR, nan=0.0) - np.nan_to_num(cL, nan=0.0)   # + = RIGHT stim
side = np.sign(signed_contrast)                              # +1 right, -1 left, 0 at 0% contrast
abs_contrast = np.abs(signed_contrast)
rt = tt["firstMovement_times"].to_numpy(float) - tt["stimOn_times"].to_numpy(float)  # reaction time

# raw-count tensors (per region) + continuous spike trains (per region; for rasters / pynapple)
data3 = preprocess_session(probe_loads, event_times, cfg)
bc = data3["bin_centers"]; bin_s = cfg.bin_size_s
region_trains = {r: region_spike_trains(probe_loads, r) for r in REGIONS_3}
print("\ntrials:", len(tt), " | contrast levels (%):", (np.unique(abs_contrast) * 100).round(2))


In [ ]:
# §7 — population PSTH per area (juxtaposed), each with a +/-1 SEM-ACROSS-TRIALS band.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for ax, r in zip(axes, REGIONS_3):
    counts = data3["regions"][r]["counts"]              # (trials, bins, units), raw integer counts
    n_tr, n_bin, n_u = counts.shape
    pop_rate = counts.mean(axis=2) / bin_s              # per-trial population rate (trials, bins)
    psth = pop_rate.mean(axis=0)                        # mean over trials AND neurons
    sem = pop_rate.std(axis=0, ddof=1) / np.sqrt(n_tr)  # <-- SEM ACROSS TRIALS (std / sqrt n_trials)
    col = region_color(r)
    ax.fill_between(bc, psth - sem, psth + sem, color=col, alpha=0.30, linewidth=0)
    ax.plot(bc, psth, color=col, lw=1.8, path_effects=stroke())
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(r, fontweight="bold")
    ax.set_xlabel("time from stimOn (s)")
    # write, per area, WHAT the band is computed over -- an error band is meaningless otherwise
    ax.text(0.03, 0.97,
            f"shaded = ±1 SEM\nacross {n_tr} trials\n(pop. rate = mean\nover {n_u} units)",
            transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec=col, alpha=0.9))
axes[0].set_ylabel("population rate (sp/s)")
fig.suptitle(f"Population PSTH per area · session {EID[:8]} · aligned to stimulus onset",
             y=1.05, fontweight="bold")
fig.text(0.5, -0.03,
    "Mean firing rate across trials and neurons, one panel per area (note the different y-scales — "
    "VISp is far sparser). Dashed line = stimulus onset. SCm ramps and stays elevated; MRN gives an "
    "early transient (~150-200 ms, near the median reaction time); VISp is weak in this session.",
    ha="center", va="top", fontsize=9, style="italic")
fig.tight_layout()
print("saved:", savefig(fig, "smoketest", "psth_per_area"))


## 8. Spike rasters — look at the raw data two ways

A PSTH is already an average. Before trusting it, look at the spikes themselves, in two complementary
cuts:

- **neurons × time, one trial** (§8a) — the whole population in a *single* decision. Shows how many
  units are active and how dense each area is, side by side.
- **trials × time, one neuron** (§8b) — one unit across *every* trial, rows **sorted by reaction
  time** (via pynapple's `compute_perievent`). This is the cut that tells *stimulus-locked* from
  *movement-locked*: a sensory response stays a vertical band at t = 0, a motor one bends along the
  reaction-time curve.


In [ ]:
# §8a — neurons x time, one example trial per area (raw spike times via region_event_spikes)
ex = int(np.flatnonzero((abs_contrast == 1.0) & (tt["feedbackType"].to_numpy(float) == 1)
                        & np.isfinite(rt) & (rt > 0.1) & (rt < 0.5))[0])   # a clean high-contrast trial
evsp = {r: region_event_spikes(probe_loads, r, event_times, pre_s=cfg.pre_s, post_s=cfg.post_s)
        for r in REGIONS_3}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for ax, r in zip(axes, REGIONS_3):
    ti, ui, tr_rel = evsp[r]                            # trial idx, unit idx, spike time (rel. to stimOn)
    sel = ti == ex
    col = region_color(r)
    ax.plot(tr_rel[sel], ui[sel], "|", color=col, markersize=4, mew=0.8)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.axvline(rt[ex], color="0.35", lw=1.2)            # this trial's movement onset
    ax.set(title=r, xlabel="time from stimOn (s)", xlim=(-0.4, 0.8),
           ylim=(-1, int(ui.max()) + 1))
axes[0].set_ylabel("neuron (unit #)")
fig.suptitle(f"Population raster — one example trial (#{ex}, 100% contrast, correct) · session {EID[:8]}",
             y=1.02, fontweight="bold")
fig.text(0.5, -0.04,
    "Each tick is one spike; one column per area, same trial. Dashed line = stimulus onset, grey "
    "line = this trial's movement onset. VISp fires sparsely, SCm and MRN densely — the raw substrate "
    "behind the PSTHs above, in a single decision.", ha="center", va="top", fontsize=9, style="italic")
fig.tight_layout()
print("saved:", savefig(fig, "smoketest", "raster_example_trial"))


In [ ]:
# §8b — trials x time, one task-modulated example neuron per area (pynapple compute_perievent)
row_of_trial = np.argsort(np.argsort(np.where(np.isfinite(rt), rt, np.inf)))  # trial -> row (RT order)
rt_curve = np.sort(rt[np.isfinite(rt)])
ev = nap.Ts(t=event_times)   # MUST stay time-ordered: pynapple silently re-sorts timestamps


def most_modulated_unit(counts):
    """Kept-unit index with the strongest post-stim transient over a low baseline (readable raster)."""
    psth = counts.mean(0) / bin_s
    base = psth[bc < 0].mean(0); peak = psth[bc >= 0].max(0)
    baserate = counts[:, bc < 0, :].mean((0, 1)) / bin_s
    mod = np.where(baserate < 12, (peak - base) / (base + 2.0), -np.inf)  # skip ultra-dense units
    return int(mod.argmax())


fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True)
for ax, r in zip(axes, REGIONS_3):
    counts = data3["regions"][r]["counts"]
    j = most_modulated_unit(counts)
    orig = int(data3["regions"][r]["keep_idx"][j])          # -> key into region_trains
    base_hz = counts[:, bc < 0, j].mean() / bin_s
    peri = nap.compute_perievent(nap.Ts(t=region_trains[r][orig]), ev, (-0.4, 0.8))
    col = region_color(r)
    for trial_i, key in enumerate(peri.keys()):
        tsp = peri[key].t
        if len(tsp):
            ax.plot(tsp, np.full_like(tsp, row_of_trial[trial_i]), "|",
                    color=col, markersize=1.8, mew=0.5, rasterized=True)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.plot(rt_curve, np.arange(rt_curve.size), color="black", lw=1.6)  # movement onset
    ax.set(title=f"{r} — unit {orig} ({base_hz:.1f} Hz baseline)",
           xlabel="time from stimOn (s)", xlim=(-0.4, 0.8))
axes[0].set_ylabel("trial (sorted by RT)")
fig.suptitle(f"Single-neuron raster — trials sorted by reaction time (pynapple compute_perievent) · "
             f"session {EID[:8]}", y=1.02, fontweight="bold")
fig.text(0.5, -0.04,
    "One task-modulated unit per area. Rows = trials, sorted by reaction time (black curve = "
    "movement onset). All three responses BEND with the black curve → they are movement-locked, not "
    "stimulus-locked: over this window the areas fire with the action, not the stimulus.",
    ha="center", va="top", fontsize=9, style="italic")
fig.tight_layout()
print("saved:", savefig(fig, "smoketest", "raster_example_neuron"))


In [ ]:
# §8c — population PSTH per area, ALL trials pooled (mean over neurons & trials, ±1 SEM across trials)
# The averaged complement to the single-neuron rasters above: every usable trial contributes to ONE
# curve per area, with NO stimulus/choice/contrast split. (Same computation as §7, repeated here so
# the raw single-neuron cuts and their population average sit together.)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for ax, r in zip(axes, REGIONS_3):
    counts = data3["regions"][r]["counts"]              # (trials, bins, units), raw integer counts
    n_tr, n_bin, n_u = counts.shape
    pop_rate = counts.mean(axis=2) / bin_s              # per-trial population rate (trials, bins)
    psth = pop_rate.mean(axis=0)                        # mean over trials AND neurons (all trials pooled)
    sem = pop_rate.std(axis=0, ddof=1) / np.sqrt(n_tr)  # +/-1 SEM ACROSS TRIALS (std / sqrt n_trials)
    col = region_color(r)
    ax.fill_between(bc, psth - sem, psth + sem, color=col, alpha=0.30, linewidth=0)
    ax.plot(bc, psth, color=col, lw=1.8, path_effects=stroke())
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(r, fontweight="bold")
    ax.set_xlabel("time from stimOn (s)")
    # state what the band is over -- an error band is meaningless without it (CLAUDE.md convention)
    ax.text(0.03, 0.97, f"shaded = +/-1 SEM\nacross {n_tr} trials\n({n_u} units pooled)",
            transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec=col, alpha=0.9))
axes[0].set_ylabel("population rate (sp/s)")
fig.suptitle(f"Population PSTH per area -- all trials pooled (no condition split) * session {EID[:8]}",
             y=1.05, fontweight="bold")
fig.text(0.5, -0.03,
    "Firing rate averaged over neurons and ALL trials (no stimulus/choice/contrast split), one panel "
    "per area (note the different y-scales -- VISp is far sparser). Shaded = +/-1 SEM across trials; "
    "dashed line = stimulus onset. The population-average complement to the single-neuron rasters above.",
    ha="center", va="top", fontsize=9, style="italic")
fig.tight_layout()
print("saved:", savefig(fig, "smoketest", "psth_per_area_pooled"))


## 9. Two model-free plots from **pynapple**

`pynapple` (the time-series toolkit demoed at the hackathon) is the project's exploration layer: it
gives rasters, correlograms and tuning curves cheaply and independently cross-checks our own binning
— verified **bit-for-bit** against `build_matrices` (worklog 2026-07-12). The single-neuron raster
above already used its `compute_perievent`; two more of its plots earn a place in the story:

- **SCm ↔ MRN cross-correlogram** — the cheap, model-free version of *"do SCm and MRN talk to each
  other?"*. A symmetric peak at zero lag = **common drive**; a shifted peak would be a lead–lag.
- **Contrast tuning** — mean evoked rate vs signed contrast, per area. A sensory area should track
  |contrast|; a motor area should be flatter or side-biased.


In [ ]:
# §9a — SCm <-> MRN cross-correlogram (pynapple compute_crosscorrelogram). ~30-40 s over all pairs.
fig = region_crosscorrelogram(region_trains["SCm"], region_trains["MRN"], "SCm", "MRN",
                              binsize=0.005, windowsize=0.2)
fig.text(0.5, -0.02,
    "Mean over all SCm-MRN unit pairs (norm=True; 1.0 = chance). A sharp peak at ZERO lag on a flat "
    "baseline = common drive, not a directed SCm->MRN exchange. This is why a joint multi-region "
    "model (Phase 3) is needed to argue direction — millisecond spike timing alone cannot.",
    ha="center", va="top", fontsize=9, style="italic")
print("saved:", savefig(fig, "smoketest", "crosscorr_SCm_MRN"))


In [ ]:
# §9b — single-neuron contrast tuning per area (pynapple helper)
fig = contrast_tuning(region_trains, event_times, signed_contrast, resp_window=(0.05, 0.15),
                      title="Single-neuron contrast tuning (evoked 50-150 ms)")
fig.text(0.5, -0.02,
    "Mean evoked rate vs signed contrast (<0 left, >0 right), averaged over each area's units. "
    "SCm rises toward both high-|contrast| ends (tracks stimulus strength); MRN is flatter; VISp "
    "is sparse. The single-neuron-rate complement to the population decoder below.",
    ha="center", va="top", fontsize=9, style="italic")
print("saved:", savefig(fig, "smoketest", "contrast_tuning"))


## 10. Decoding: stimulus vs choice tuning of each area

A population decoder is the compact way to ask *what each area's activity carries* — and it doubles
as an answer to the most common trap in this task: *"decoding accuracy doesn't move with coherence
the way it should."* That comes from decoding **the wrong variable**.

**What "coherence" is here.** IBL does not use random-dot coherence; it uses a Gabor of varying
**contrast** ∈ {0, 6.25, 12.5, 25, 100} %. Contrast is the rodent analogue of coherence — the
strength of the *sensory evidence*. On each trial the Gabor appears on the **left or right** and
the mouse turns a wheel to bring it to centre. The **choice** is the wheel-turn direction.

**Two different things you can decode, and they depend on contrast oppositely:**

| decode | what it is | expected dependence on contrast |
|---|---|---|
| **stimulus side** | which side the Gabor was on | **rises** with contrast — it *is* the sensory signal |
| **choice** | which way the mouse actually turned | **need not rise** — the mouse moves on every trial |

The key fact: **the mouse commits to a movement on every single trial, including at 0 % contrast**
(there it guesses, biased by the block prior `probabilityLeft`). Motor-related areas (SCm, MRN)
encode the *executed action*, so **choice is decodable at all contrasts — even when there is no
stimulus to see.** If you decode *choice* and expect its accuracy to climb with contrast like a
sensory curve, it will not, and nothing is broken. The variable whose decoding climbs with
contrast is the **stimulus**, not the choice.

Two things sharpen the confusion, and the code below handles both:

1. **At high contrast, choice ≈ stimulus** (the mouse is ~always correct), so the two decoders
   *converge* — which can masquerade as "choice tracks contrast". At **low** contrast they
   *diverge*: stimulus decoding falls to chance while choice decoding stays high. Watch the gap.
2. **Class imbalance inflates plain accuracy.** At high contrast one choice dominates, so a decoder
   that always guesses the majority scores ~0.8 for free. We therefore report **balanced accuracy**
   under **stratified** cross-validation; chance = 0.5.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# choice == -1 => mouse chose the RIGHT stimulus (verified: at |c|>=50% choice=-1 <-> side=+1)
chose_right = (choice == -1).astype(int)
go = choice != 0                       # drop no-go trials for the choice decoder


def window_features(counts, t0, t1):
    """Mean spike count per neuron over the window [t0, t1) s -> (trials, neurons)."""
    sel = (bc >= t0) & (bc < t1)
    return counts[:, sel, :].mean(axis=1)


def decode(X, y, n_splits=5):
    """Balanced accuracy of an L2-logistic decoder under stratified CV. Returns (mean, sd).

    Balanced accuracy + class_weight='balanced' so the decoder cannot win by always guessing
    the majority class (the trap at high contrast). NaN if a class is too small to stratify.
    """
    y = np.asarray(y)
    classes, n = np.unique(y, return_counts=True)
    if len(classes) < 2 or n.min() < n_splits:
        return np.nan, np.nan
    clf = make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced"))
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=PARAMS["random_seed"])
    s = cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy")
    return s.mean(), s.std()


STIM_WIN, CHOICE_WIN = (0.05, 0.20), (0.0, 0.40)   # early sensory / broad (spans movement onset)
levels = np.unique(abs_contrast); lev_pct = levels * 100

# 2x2: psychometric + one decoding panel per area (decluttered; explanation is in the caption below)
fig, axes = plt.subplots(2, 2, figsize=(11, 7.4))
axp = axes[0, 0]
sc = np.unique(signed_contrast)
axp.plot(sc * 100, [np.mean(chose_right[signed_contrast == s]) for s in sc], "-o",
         color=OKABE_ITO[4], lw=1.8, path_effects=stroke())
axp.axhline(0.5, color="k", ls=":", lw=1); axp.axvline(0, color="k", ls=":", lw=1)
axp.set(xlabel="signed contrast (%)  [+ = right]", ylabel="P(choose right)",
        title="Behaviour: psychometric", ylim=(-0.03, 1.03))

for ax, r in zip([axes[0, 1], axes[1, 0], axes[1, 1]], REGIONS_3):
    counts = data3["regions"][r]["counts"]
    Xs, Xc = window_features(counts, *STIM_WIN), window_features(counts, *CHOICE_WIN)
    sm, se, cm, ce = [], [], [], []
    for L in levels:
        m = abs_contrast == L
        a, e = decode(Xs[m & (side != 0)], side[m & (side != 0)]) if L > 0 else (np.nan, np.nan)
        a2, e2 = decode(Xc[m & go], chose_right[m & go])
        sm.append(a); se.append(e); cm.append(a2); ce.append(e2)
    ax.axhline(0.5, color="k", ls=":", lw=1)
    ax.errorbar(lev_pct, sm, yerr=se, fmt="-s", color=OKABE_ITO[4], lw=1.6, capsize=3,
                path_effects=stroke(), label="stimulus side")
    ax.errorbar(lev_pct, cm, yerr=ce, fmt="-o", color=OKABE_ITO[0], lw=1.6, capsize=3,
                path_effects=stroke(), label="choice")
    ax.set_xscale("symlog", linthresh=5); ax.set_xticks(lev_pct)
    ax.set_xticklabels([f"{v:g}" for v in lev_pct], fontsize=8)
    ax.set(title=f"{r}  ({counts.shape[2]} units)", xlabel="|contrast| (%)",
           ylabel="balanced accuracy", ylim=(0.30, 1.0))
    ax.legend(frameon=False, fontsize=8, loc="upper left")

fig.suptitle(f"Decoding stimulus vs choice at different contrast ('coherence') · session {EID[:8]}",
             y=1.0, fontweight="bold")
fig.text(0.5, -0.02,
    "Balanced accuracy, 5-fold stratified CV (chance 0.5, dotted; bars = ±1 SD across folds). "
    "Windows: stimulus [50,200] ms, choice [0,400] ms. Blue = decode which SIDE the stimulus was "
    "on (sensory); orange = decode the mouse's CHOICE (action). Stimulus decoding rises with "
    "contrast; choice stays high and flat — read the note below.",
    ha="center", va="top", fontsize=9, style="italic")
fig.tight_layout()
print("saved:", savefig(fig, "smoketest", "decoding_vs_contrast"))


### What the panels say (session 8a3a0197, 287 usable go-trials)

- **Stimulus-side decoding climbs with contrast**, exactly as the "coherence" intuition predicts:
  SCm **0.36 → 0.93** and MRN **0.56 → 0.86** from 6 % to 100 % contrast.
- **Choice decoding is high and roughly flat**, and it is **above chance already at 0 % contrast**
  (SCm/MRN ≈ 0.58 with *no stimulus at all*), rising to ~0.95 — because it reflects the *movement
  the mouse made*, not the evidence it saw.
- **The two curves diverge at low contrast and converge at high contrast** (SCm at 6 %: stimulus
  0.36 vs choice 0.81; at 100 %: 0.93 vs 0.95). That gap **is** the answer: choice-decoding
  accuracy is not supposed to track contrast.
- **VISp sits near chance here** for both labels — this single session has only 23 VISp units at
  ≥ 1 Hz and it is the project's weakest region; that is exactly why we replicate across the 5
  triplet sessions rather than trust one. (Note SCm decodes the stimulus *well*: the superior
  colliculus is itself a visual structure, so a strong visual signal there is expected.)

**Bottom line for your colleague.** If a decoder's accuracy "doesn't make sense with coherence",
check *which label* is being decoded. A **choice** decoder should **not** scale with contrast —
the mouse acts on every trial, and motor areas encode the act. A **stimulus** decoder should, and
does. The **window** matters too: an early window (≈ 50–200 ms) isolates the sensory signal; a
broad or peri-movement window is dominated by the action. (Standing project caveat: over
[0, +0.8] s movement has already begun on 92 % of trials, so any late "decision" signal is
entangled with movement — hence the separate `firstMovement`-aligned tensors.)
